# True North — SBC Kids' Bible SLM
### QLoRA fine-tune + base-vs-tuned eval  (+ optional frontier benchmark)

**First: Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Run-all will pause twice — once to **authorize Google Drive**, once for your **gateway key**. After that it's hands-off.

**All results auto-save to your Google Drive → `MyDrive/true-north/`** — they persist even if Colab disconnects, so there's nothing to manually download. 🎉

In [ ]:
!git clone https://github.com/graceyan212/bible-slm.git
%cd bible-slm
!pip install -q unsloth anthropic openai

## 1 · Mount Google Drive (results auto-save here & survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUTDIR = '/content/drive/MyDrive/true-north'
os.makedirs(OUTDIR, exist_ok=True)
print('✅ Results will save to your Drive:', OUTDIR)

## 2 · Config + judge key
Set the base model + your TrueFoundry judge. Key via the 🔑 Colab Secret `TRUEFOUNDRY_API_KEY` (set-once) or the prompt.

In [ ]:
import getpass

# --- Base model to fine-tune (free on the T4) ---
MODEL = 'unsloth/Qwen3-4B-Instruct-2507'        # faster/lighter: 'unsloth/Qwen3-1.7B-Instruct'
os.environ['BASE_MODEL']  = MODEL
os.environ['ADAPTER_OUT'] = f'{OUTDIR}/sbc-lora'   # save the trained adapter to Drive too

# --- Judge (eval scoring) via TrueFoundry gateway ---
os.environ['JUDGE_BASE_URL'] = 'https://gateway.truefoundry.ai'
os.environ['JUDGE_MODEL']    = 'claude-sonnet-5'    # confirmed working in your console
key = None
try:
    from google.colab import userdata
    key = userdata.get('TRUEFOUNDRY_API_KEY')
except Exception:
    pass
os.environ['JUDGE_API_KEY'] = key or getpass.getpass('TrueFoundry API key (user-...): ')
print('Base:', MODEL, '| judge:', os.environ['JUDGE_MODEL'], '| adapter →', os.environ['ADAPTER_OUT'])

## 2b · Quick judge check (~10 s) — catches a bad key/model before the long run

In [ ]:
model = os.environ.get('JUDGE_MODEL', 'claude-sonnet-5')
from openai import OpenAI
_c = OpenAI(api_key=os.environ.get('JUDGE_API_KEY') or os.environ.get('OPENAI_API_KEY'), base_url=os.environ['JUDGE_BASE_URL'])
_r = _c.chat.completions.create(model=model, max_tokens=20, messages=[{'role':'user','content':'Reply with exactly: OK'}])
print('✅ Judge reachable — model said:', _r.choices[0].message.content)

## 3 · Baseline eval — BEFORE training
Expect the base to **flatten** on baptism/eternal-security and **cave** under pushback.

In [ ]:
!python eval/run_eval.py --model base --hf {MODEL} --out {OUTDIR}/results_base.json

## 4 · Fine-tune (QLoRA, ~30–60 min) → adapter saved to Drive

In [ ]:
!python train/train_qlora.py

## 5 · Tuned eval — same 52 scenarios, same prompt

In [ ]:
!python eval/run_eval.py --model tuned --hf {MODEL} --adapter {OUTDIR}/sbc-lora --out {OUTDIR}/results_tuned.json

## 6 · Results table — base vs tuned (headline)

In [ ]:
!python eval/run_eval.py --compare {OUTDIR}/results_base.json {OUTDIR}/results_tuned.json --md {OUTDIR}/results_table.md
from IPython.display import Markdown, display
display(Markdown(open(f'{OUTDIR}/results_table.md').read()))

## 7 · Bonus — Frontier vs. your SLM (optional · API-only, no GPU needed)
Runs a **frontier model** through the same 52 scenarios + same judge, so you can compare it to your tuned SLM. Contestant is kept **≠ the judge** (no self-scoring). Change `GEN_MODEL` to try others (`openai-group/gpt-5`, `claude-group/claude-opus-4-8`).

In [ ]:
os.environ['GEN_MODEL'] = 'openai-group/gpt-4o'   # the frontier contestant (≠ claude-sonnet-5 judge)
!python eval/run_eval.py --model frontier --backend gateway --out {OUTDIR}/results_frontier.json
!python eval/run_eval.py --metrics {OUTDIR}/results_frontier.json

## ✅ Done — everything is in your Google Drive
Open **Drive → `MyDrive/true-north/`**: `results_base.json`, `results_tuned.json`, `results_table.md`, `results_frontier.json`, and the `sbc-lora/` adapter. Nothing to manually download. Send me `results_frontier.json` and I'll build the three-way table.

In [ ]:
print('Saved to', OUTDIR, ':')
for f in sorted(os.listdir(OUTDIR)): print('  ', f)